# provenance

> `SourceRef` — the cross-cutting provenance reference (CR-19). **Identity = `content_hash` (PRIMARY); location = `locator`; region = optional atomic typed `slice`.** `verify()` is hash-based regardless of whether the locator still resolves — the structural fix for dangling row-id provenance (cache-hit rows; ledgers E13/D3).

In [ ]:
#| default_exp provenance

## Design notes (locked pass-2 Thread 2)

- `content_hash` is over the **CONSUMED (sliced) content**, tagged `"algo:hexdigest"` via `cjm_plugin_system.utils.hashing` (a scalar kept as a string by principled exception — nothing slices into a hash).
- The slice is BY DEFINITION into the resource the locator addresses; where a target has multiple content facets, the slice KIND selects which (`TimeSlice` → audio, `CharSlice` → text). No separate frame field.
- There is deliberately **no `derived_from` field**: the provenance chain IS the graph edges (`Segment → Transcript → AudioSegment → Source`); a per-ref chain field would duplicate topology (composition is edges).
- Constructed by the **composition layer** (workflow cores / CR-18), never by capabilities — the placement rationale for this library (data-nouns vs storage-verbs).

In [ ]:
#| export
from dataclasses import dataclass
from typing import Any, Dict, Optional

from cjm_plugin_system.core.errors import PluginInputError
from cjm_plugin_system.utils.hashing import hash_bytes, verify_hash

from cjm_context_graph_primitives.locators import ResourceLocator, locator_from_dict
from cjm_context_graph_primitives.slices import TypedSlice, slice_from_dict

In [ ]:
#| export
@dataclass(frozen=True)
class SourceRef:
    """A provenance reference to (a region of) a resource.

    Replaces the old plugin_name/table_name/row_id/segment_slice shape: the
    locator is a structured sum type (no `external:<path>` string abuse), the
    hash is primary identity, and the slice is typed and atomic.
    """
    locator: ResourceLocator           # WHERE the content lives/lived (may dangle without breaking verifiability)
    content_hash: str                  # PRIMARY identity: "algo:hexdigest" over the CONSUMED (sliced) content
    slice: Optional[TypedSlice] = None  # Consumed region, framed to the locator target; None = whole resource

    def to_uri(self) -> str:  # Canonical string: "<locator-uri>[#<slice-string>]@<content_hash>"
        """Render the complete canonical string form (grep-able by locator, slice, or hash)."""
        base = self.locator.to_uri()
        if self.slice is not None:
            base = f"{base}#{self.slice.to_slice_string()}"
        return f"{base}@{self.content_hash}"

    def __str__(self) -> str:  # Same as `to_uri`
        return self.to_uri()

    def to_dict(self) -> Dict[str, Any]:  # Nested wire dict
        """Serialize to the wire dict form."""
        return {
            "locator": self.locator.to_dict(),
            "content_hash": self.content_hash,
            "slice": self.slice.to_dict() if self.slice is not None else None,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict with nested locator/slice dicts
    ) -> "SourceRef":  # Reconstructed reference (unknown locator/slice kinds round-trip)
        """Reconstruct from the wire dict form."""
        if "locator" not in d or "content_hash" not in d:
            raise PluginInputError(
                f"SourceRef dict missing required keys: {sorted(d.keys())!r}",
                fields_invalid=[k for k in ("locator", "content_hash") if k not in d],
            )
        sl = d.get("slice")
        return cls(
            locator=locator_from_dict(d["locator"]),
            content_hash=d["content_hash"],
            slice=slice_from_dict(sl) if sl is not None else None,
        )

    def verify(
        self,
        current_content: bytes  # Current bytes of the CONSUMED (sliced) content
    ) -> bool:  # True if content still matches the stored hash
        """Hash-verify content — works even when the locator no longer resolves."""
        return verify_hash(current_content, self.content_hash)

    @staticmethod
    def compute_hash(
        content: bytes,       # Content to hash
        algo: str = "sha256"  # Hash algorithm name
    ) -> str:  # Hash string in "algo:hexdigest" format
        """Compute a content hash for use in a SourceRef."""
        return hash_bytes(content, algo)

In [ ]:
from cjm_context_graph_primitives.locators import GraphNodeRef, FileRef, UnknownLocator
from cjm_context_graph_primitives.slices import CharSlice, TimeSlice

text = "It's one small step for man,"
ref = SourceRef(
    locator=FileRef(path="/runs/run_x.json"),
    content_hash=SourceRef.compute_hash(text.encode()),
    slice=CharSlice(25, 53),
)

# identity-vs-location split: verify() works regardless of locator resolution
assert ref.verify(text.encode())
assert not ref.verify(b"tampered")

# canonical render carries locator, slice, and hash
assert ref.to_uri().startswith("file:/runs/run_x.json#char:25-53@sha256:")

# wire round-trip
assert SourceRef.from_dict(ref.to_dict()) == ref

# slice=None (whole resource) round-trips
whole = SourceRef(locator=GraphNodeRef(node_id="n1"), content_hash="sha256:00")
assert whole.to_dict()["slice"] is None and SourceRef.from_dict(whole.to_dict()) == whole
assert whole.to_uri() == "graph-node:n1@sha256:00"

# unknown locator kind inside a SourceRef round-trips losslessly
fut = {"locator": {"kind": "actor", "actor_id": "agent:claude"},
       "content_hash": "sha256:ab", "slice": None}
r2 = SourceRef.from_dict(fut)
assert isinstance(r2.locator, UnknownLocator) and r2.to_dict() == fut

# kind-selects-facet: same node, two facets, two refs
audio_ref = SourceRef(locator=GraphNodeRef(node_id="seg-9"), content_hash="sha256:aa", slice=TimeSlice(6.6, 9.8))
text_ref = SourceRef(locator=GraphNodeRef(node_id="seg-9"), content_hash="sha256:bb", slice=CharSlice(0, 28))
assert audio_ref != text_ref and audio_ref.locator == text_ref.locator

# missing keys raise
try:
    SourceRef.from_dict({"content_hash": "sha256:00"})
    raise AssertionError("should have raised")
except PluginInputError:
    pass